| # | Synapse Schema | Synapse Table | Databricks Table |
|---|---|---|---|
| 1 | dal_v | vw_udlpisystems_opsdata_mis_rtit_headgrade | opsdata_mis_rtit_headgrade |
| 2 | dal_v | vw_udlpisystems_opsdata_cs_mis_rtit_sandmetrics | opsdata_mis_rtit_sandmetrics |
| 3 | dal_v | vw_udlpisystems_opsdata_cs_pm_reporting_data | opsdata_pm_reporting_data |
| 4 | dal_v | vw_udlpisystems_opsdata_cs_pm_reporting_recovery_data | opsdata_pm_reporting_recovery_data |
| 5 | dal_v | vw_udlpisystems_pm_reporting_locations_hierarchy | globalpi_pm_reporting_locations_hierarchy |

# Change Data Feed (CDF) Guide
## Schema: `lakehouse.analytics_and_enablement_staging_pi_systems`

---

## Architecture Overview

Tables in this schema follow a **two-catalog pattern** managed by the RTLH (Rio Tinto Lakehouse) framework:

| Catalog | Purpose | Table Type |
|---------|---------|------------|
| `lakehouse` | Current state (latest snapshot) | VIEW over `_lakehouse_aws_syd` |
| `lakehouse_history` | Change history (CDF) | VIEW over `_lakehouse_aws_syd_history` |

Both catalogs expose **views** that wrap the physical Delta tables in internal catalogs (`_lakehouse_aws_syd` and `_lakehouse_aws_syd_history`). Users never access the internal catalogs directly.

---

## Tables and Their CDF Counterparts

| Current State Table (`lakehouse.*`) | CDF Table (`lakehouse_history.*`) | CDF Status |
|--------------------------------------|-------------------------------------|------------|
| `opsdata_mis_rtit_headgrade` | `opsdata_mis_rtit_headgrade_cdf` | Active (562 changes, Jun 1–7) |
| `opsdata_mis_rtit_sandmetrics` | `opsdata_mis_rtit_sandmetrics_cdf` | Active (2,539 changes, Jun 1–7) |
| `opsdata_pm_reporting_data` | `opsdata_pm_reporting_data_cdf` | Active (2,169 changes, Jun 1–7) |
| `opsdata_pm_reporting_recovery_data` | `opsdata_pm_reporting_recovery_data_cdf` | Active (101 changes, Jun 2–5) |
| `globalpi_pm_reporting_locations_hierarchy` | `globalpi_pm_reporting_locations_hierarchy_cdf` | Enabled, no recent changes |

All 5 tables have `delta.enableChangeDataFeed = true` on the underlying Delta tables (confirmed via functioning CDF views in `lakehouse_history`). Every table in `lakehouse.analytics_and_enablement_staging_pi_systems` has a 1:1 CDF companion in `lakehouse_history.analytics_and_enablement_staging_pi_systems`.

---

## Column Structure

### Current State Tables (`lakehouse.*`)
- Business columns (e.g. `actual`, `budget`, `datefrom`, `dateto`, `sitecode`, etc.)
- `rtlh_ingestion_time` — timestamp when the row was ingested
- `rtlh_row_hash` — hash of the row used for change detection

### CDF Tables (`lakehouse_history.*`)
Same business columns as above, plus:
- `_change_type` — Delta CDF metadata: only `insert` and `delete` are used (the RTLH framework does NOT produce `update_preimage`/`update_postimage` — updates are implemented as delete-old + insert-new pairs)
- `_commit_timestamp` — when the change was committed to the Delta table
- `_commit_version` — the Delta table version number of the commit
- `rtlh_cdc_operation` — RTLH-specific CDC operation marker
- `rtlh_ingestion_time` — ingestion timestamp
- `rtlh_job_run_id` — the job run that produced this change
- `rtlh_row_hash` — row hash for deduplication

---

## How CDF Works

1. **Ingestion**: The RTLH framework ingests data into the physical Delta tables in `_lakehouse_aws_syd`. It uses `rtlh_row_hash` to detect changes compared to the previous state.

2. **Update strategy — delete + insert**: The framework does NOT use in-place UPDATEs. When a row changes, the old version is deleted and a new version is inserted. Consequently, the CDF table only contains `_change_type` values of `insert` and `delete` — never `update_preimage` or `update_postimage`.

3. **Delta CDF**: The underlying Delta tables have `delta.enableChangeDataFeed = true`. Each delete + insert pair is recorded with metadata (`_change_type`, `_commit_timestamp`, `_commit_version`).

4. **CDF Views**: The `lakehouse_history` catalog exposes these changes via views that read from the internal `_lakehouse_aws_syd_history` tables, making the change feed accessible without direct access to internal catalogs.

5. **CDF retention**: The CDF history is NOT a full historical record — it only retains recent changes (observed: ~1 week of history). Do not rely on it for long-range auditing.

---

## Querying CDF

### Get recent changes (last 24 hours)
```sql
SELECT *
FROM lakehouse_history.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade_cdf
WHERE _commit_timestamp > current_timestamp() - INTERVAL 1 DAY
ORDER BY _commit_timestamp DESC
```

### Get changes by operation type
```sql
SELECT _change_type, COUNT(*) as change_count
FROM lakehouse_history.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade_cdf
WHERE _commit_timestamp > '2026-06-01'
GROUP BY _change_type
```

### Track a specific record's history
```sql
SELECT *
FROM lakehouse_history.analytics_and_enablement_staging_pi_systems.opsdata_mis_rtit_headgrade_cdf
WHERE tagname = 'your_tag_name'
ORDER BY _commit_timestamp
```

---

## Known Data Quality Issue: Exact Duplicates

Several tables in this schema contain **exact duplicate rows** (identical `rtlh_row_hash`, same `rtlh_ingestion_time`) loaded during the initial batch (2026-02-16). This is a **loading bug**, not standard behaviour.

| Table | Total Rows | Unique Hashes | Exact Duplicate Rows | Severity |
|-------|-----------|--------------|---------------------|----------|
| `opsdata_mis_rtit_headgrade` | 12,442 | 12,442 | 0 (0.00%) | Clean |
| `opsdata_mis_rtit_sandmetrics` | 99,154 | 85,569 | 13,585 (13.70%) | High |
| `opsdata_pm_reporting_data` | 157,938 | 155,746 | 2,192 (1.39%) | Low |
| `opsdata_pm_reporting_recovery_data` | 2,626 | 2,527 | 99 (3.77%) | Medium |
| `globalpi_pm_reporting_locations_hierarchy` | 3,318 | 3,318 | 0 (0.00%) | Clean |

3 out of 5 tables are affected. The duplicates appear 2x or 3x per hash — likely caused by a faulty JOIN in the source query during initial load. Subsequent incremental batches did not introduce new duplicates.

### Duplicate distribution (`opsdata_mis_rtit_sandmetrics`)

| Copies per hash | Number of hashes | Rows contributed |
|-----------------|-----------------|-----------------|
| 1 (unique) | 74,964 | 74,964 |
| 2 (duplicated) | 7,625 | 15,250 |
| 3 (triplicated) | 2,980 | 8,940 |
| **Total** | **85,569** | **99,154** |

### Root cause

The exact duplicates are NOT caused by CDF or multi-version changes. They are identical rows (same hash, same ingestion timestamp) loaded multiple times within a single batch. The RTLH framework uses `rtlh_row_hash` for change detection and should never produce exact copies — this points to a faulty source query (likely a bad JOIN or missing DISTINCT) in the initial data load pipeline.

### Deduplication pattern
```sql
-- Remove exact duplicates (same rtlh_row_hash)
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY rtlh_row_hash ORDER BY rtlh_ingestion_time DESC) AS rn
  FROM lakehouse.analytics_and_enablement_staging_pi_systems.<table>
) WHERE rn = 1
```

For `opsdata_pm_reporting_data`, there are also **natural key duplicates** (same `elementattribute + starttimeutc`, different hashes) representing data corrections. To get the latest version per logical record:
```sql
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY elementattribute, starttimeutc
    ORDER BY rtlh_ingestion_time DESC
  ) AS rn
  FROM lakehouse.analytics_and_enablement_staging_pi_systems.opsdata_pm_reporting_data
) WHERE rn = 1
```

**Important**: You do NOT need the CDF table to deduplicate. The `lakehouse` current-state table already contains the latest data — deduplication is purely a consumer-side operation on `rtlh_row_hash` or natural key.

---

## Key Differences from `_ogr_azr_syd` Tables

Tables in `_ogr_azr_syd` (e.g. `gensd03_sicom` schema) are **direct MANAGED Delta tables** — not views. They:
- Have the same `rtlh_row_hash` and `rtlh_ingestion_time` columns
- Do NOT have pre-built CDF companion views
- Require using `table_changes()` function directly (if CDF is enabled)
- Live in a raw/internal catalog without the `lakehouse`/`lakehouse_history` facade

---

## Summary

The `lakehouse` pattern provides a clean separation:
- **Read current data** → query `lakehouse.analytics_and_enablement_staging_pi_systems.<table>`
- **Read change history** → query `lakehouse_history.analytics_and_enablement_staging_pi_systems.<table>_cdf`

No need to enable CDF manually or use `table_changes()` — the RTLH framework handles this automatically and exposes it via the `lakehouse_history` catalog.

---

## When to Use CDF vs Current State

| Use Case | Table to Query |
|----------|---------------|
| Get latest data (reporting, analytics) | `lakehouse.*` (with dedup if needed) |
| Identify what changed since last run | `lakehouse_history.*_cdf` |
| Incremental downstream processing | `lakehouse_history.*_cdf` filtered by `_commit_timestamp` |
| Full historical audit trail | NOT supported — CDF retention is limited (~1 week) |
| Deduplicate exact duplicates | `lakehouse.*` using `ROW_NUMBER() OVER (PARTITION BY rtlh_row_hash ...)` |
| Get latest version per natural key | `lakehouse.*` using `ROW_NUMBER() OVER (PARTITION BY <natural_key> ORDER BY rtlh_ingestion_time DESC)` |


In [1]:
from databricks import sql
import polars as pl
import pyodbc

# Synapse Connection
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = ''
synapse_driver = '{ODBC Driver 17 for SQL Server}'
synapse_schema_name = 'HSTG_V'
synapse_table_name = 'VW_HSTG_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""
synapse_conn = pyodbc.connect(synapse_connection_string)

# Databricks Connection
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'

databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token
)

In [2]:
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [3]:
def compare_synapse_vs_databricks(
    databricks_conn,
    synapse_conn,
    synapse_schema_name,
    synapse_table_name,
    databricks_catalog_name,
    databricks_schema_name,
    databricks_table_name,
    candidate_keys=None,
    float_round_decimals=(6, 3),
    include_diagnostics=True,
):
    # Type normalization helpers
    def base_type(dtype: str) -> str:
        return str(dtype).split("(")[0].strip().upper() if dtype is not None else None

    databricks_to_synapse_type_map = {
        "TINYINT": "TINYINT",
        "SMALLINT": "SMALLINT",
        "INT": "INT",
        "INTEGER": "INT",
        "BIGINT": "BIGINT",
        "FLOAT": "REAL",
        "REAL": "REAL",
        "DOUBLE": "FLOAT",
        "DECIMAL": "DECIMAL",
        "NUMERIC": "NUMERIC",
        "STRING": "VARCHAR",
        "VARCHAR": "VARCHAR",
        "CHAR": "CHAR",
        "BINARY": "VARBINARY",
        "DATE": "DATE",
        "TIMESTAMP": "DATETIME2",
        "TIMESTAMP_NTZ": "DATETIME2",
        "TIMESTAMP_LTZ": "DATETIMEOFFSET",
        "BOOLEAN": "BIT",
    }

    # 1) Read schema metadata
    synapse_schema_query = f"""
        select
            upper(column_name) as column_name,
            upper(data_type) as synapse_data_type
        from information_schema.columns
        where table_schema = '{synapse_schema_name}'
          and table_name = '{synapse_table_name}'
          and column_name not like 'OMD%'
          and column_name not like 'HSTG%'
        order by ordinal_position
    """

    databricks_schema_query = f"""
        select
            upper(column_name) as column_name,
            upper(data_type) as databricks_data_type
        from {databricks_catalog_name}.information_schema.columns
        where table_schema = '{databricks_schema_name}'
          and table_name = '{databricks_table_name}'
          and column_name not like 'rtlh%'
    """

    synapse_schema_df = pl.read_database(synapse_schema_query, synapse_conn)
    databricks_schema_df = pl.read_database(databricks_schema_query, databricks_conn)

    # Map Databricks type -> expected Synapse type
    databricks_schema_df = databricks_schema_df.with_columns(
        pl.col("databricks_data_type")
        .map_elements(lambda x: databricks_to_synapse_type_map.get(base_type(x), None), return_dtype=pl.Utf8)
        .alias("synapse_expected_type")
    )

    # 2) Compare schema
    schema_comparison_df = synapse_schema_df.join(
        databricks_schema_df,
        on="column_name",
        how="full"
    )

    schema_mismatches_df = schema_comparison_df.filter(
        pl.col("synapse_data_type").is_null()
        | pl.col("databricks_data_type").is_null()
        | (
            pl.col("synapse_data_type").map_elements(base_type, return_dtype=pl.Utf8)
            != pl.col("synapse_expected_type").map_elements(base_type, return_dtype=pl.Utf8)
        )
    )

    schema_matches = schema_mismatches_df.height == 0

    # 3) Row count check
    synapse_count_query = f"""
        select count(*) as row_count
        from {synapse_schema_name}.{synapse_table_name}
        where omd_current_record_indicator = 'Y'
          and omd_deleted_record_indicator = 'N'
    """

    databricks_count_query = f"""
        select count(*) as row_count
        from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
    """

    synapse_row_count = int(pl.read_database(synapse_count_query, synapse_conn)["row_count"][0])
    databricks_row_count = int(pl.read_database(databricks_count_query, databricks_conn)["row_count"][0])
    row_count_matches = synapse_row_count == databricks_row_count

    # 4) Full match on common columns
    common_cols = sorted(
        set(synapse_schema_df["column_name"].to_list())
        & set(databricks_schema_df["column_name"].to_list())
    )

    if not common_cols:
        return {
            "schema_matches": schema_matches,
            "row_count_matches": row_count_matches,
            "full_match": False,
            "reason": "No common columns found for full comparison.",
            "synapse_row_count": synapse_row_count,
            "databricks_row_count": databricks_row_count,
            "schema_mismatches_df": schema_mismatches_df,
            "schema_comparison_df": schema_comparison_df,
            "matched_rows": 0,
            "pct_of_synapse": 0.0,
            "pct_of_databricks": 0.0,
            "pct_overall": 0.0,
            "diagnostics": {
                "hash_scenarios_df": pl.DataFrame([]),
                "duplicate_profile_df": pl.DataFrame([]),
                "key_overlap_df": pl.DataFrame([]),
                "key_multiplicity_df": pl.DataFrame([]),
                "float_columns": [],
                "string_columns": [],
            },
        }

    synapse_cols_sql = ", ".join([f"[{c}] as [SYNAPSE_{c}]" for c in common_cols])
    databricks_cols_sql = ", ".join([f"`{c}` as `DATABRICKS_{c}`" for c in common_cols])

    synapse_full_query = f"""
        select {synapse_cols_sql}
        from {synapse_schema_name}.{synapse_table_name}
        where omd_current_record_indicator = 'Y'
          and omd_deleted_record_indicator = 'N'
    """

    databricks_full_query = f"""
        select {databricks_cols_sql}
        from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
    """

    databricks_dedup_query = f"""
        with ranked as (
            select *,
                   row_number() over (
                       partition by rtlh_row_hash
                       order by rtlh_ingestion_time desc
                   ) as rn
            from {databricks_catalog_name}.{databricks_schema_name}.{databricks_table_name}
        )
        select {databricks_cols_sql}
        from ranked
        where rn = 1
    """

    synapse_full_table_df = pl.read_database(synapse_full_query, synapse_conn)
    databricks_full_table_df = pl.read_database(databricks_full_query, databricks_conn)
    databricks_dedup_full_table_df = pl.read_database(databricks_dedup_query, databricks_conn)

    # Normalize timezone-aware datetime columns in Databricks data
    tz_cols = [
        name for name, dtype in databricks_full_table_df.schema.items()
        if dtype == pl.Datetime and getattr(dtype, "time_zone", None) is not None
    ]
    if tz_cols:
        databricks_full_table_df = databricks_full_table_df.with_columns(
            [pl.col(c).dt.replace_time_zone(None) for c in tz_cols]
        )

    # Normalize timezone-aware datetime columns in deduplicated Databricks data
    dedup_tz_cols = [
        name for name, dtype in databricks_dedup_full_table_df.schema.items()
        if dtype == pl.Datetime and getattr(dtype, "time_zone", None) is not None
    ]
    if dedup_tz_cols:
        databricks_dedup_full_table_df = databricks_dedup_full_table_df.with_columns(
            [pl.col(c).dt.replace_time_zone(None) for c in dedup_tz_cols]
        )

    INT_LIKE = {
        pl.Boolean, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64
    }
    FLOAT_LIKE = {pl.Float32, pl.Float64}

    def normalize_types(df: pl.DataFrame) -> pl.DataFrame:
        exprs = []
        for c, d in df.schema.items():
            if d in INT_LIKE:
                exprs.append(pl.col(c).cast(pl.Int64).alias(c))
            elif d in FLOAT_LIKE:
                exprs.append(pl.col(c).cast(pl.Float64).alias(c))
            elif str(d).startswith("Datetime"):
                e = pl.col(c)
                if "time_zone" in str(d):
                    e = e.dt.replace_time_zone(None)
                exprs.append(e.dt.cast_time_unit("us").alias(c))
        return df.with_columns(exprs) if exprs else df

    def normalize_strings(df: pl.DataFrame, string_cols, mode: str) -> pl.DataFrame:
        if not string_cols:
            return df
        if mode == "trim":
            return df.with_columns([pl.col(c).str.strip_chars().alias(c) for c in string_cols])
        if mode == "trim_upper":
            return df.with_columns([pl.col(c).str.strip_chars().str.to_uppercase().alias(c) for c in string_cols])
        return df

    def round_float_cols(df: pl.DataFrame, float_cols, decimals: int) -> pl.DataFrame:
        if not float_cols:
            return df
        return df.with_columns([pl.col(c).round(decimals).alias(c) for c in float_cols])

    def add_row_hash(df: pl.DataFrame) -> pl.DataFrame:
        return df.with_columns(pl.struct(pl.all()).hash().alias("row_hash"))

    def calc_hash_match_stats(left_df: pl.DataFrame, right_df: pl.DataFrame) -> dict:
        left_h = add_row_hash(left_df)
        right_h = add_row_hash(right_df)

        left_counts = (
            left_h
            .group_by("row_hash")
            .len()
            .rename({"len": "left_cnt"})
        )
        right_counts = (
            right_h
            .group_by("row_hash")
            .len()
            .rename({"len": "right_cnt"})
        )

        cmp_df = (
            left_counts.join(right_counts, on="row_hash", how="full")
            .with_columns(
                pl.col("left_cnt").fill_null(0).cast(pl.Int64),
                pl.col("right_cnt").fill_null(0).cast(pl.Int64),
            )
            .with_columns(pl.min_horizontal("left_cnt", "right_cnt").alias("matched_cnt"))
        )

        left_total = int(left_counts["left_cnt"].sum()) if left_counts.height else 0
        right_total = int(right_counts["right_cnt"].sum()) if right_counts.height else 0
        matched_rows_local = int(cmp_df["matched_cnt"].sum()) if cmp_df.height else 0

        only_left = int(
            cmp_df
            .filter(pl.col("left_cnt") > pl.col("right_cnt"))
            .with_columns((pl.col("left_cnt") - pl.col("right_cnt")).alias("delta"))["delta"]
            .sum() or 0
        )
        only_right = int(
            cmp_df
            .filter(pl.col("right_cnt") > pl.col("left_cnt"))
            .with_columns((pl.col("right_cnt") - pl.col("left_cnt")).alias("delta"))["delta"]
            .sum() or 0
        )

        return {
            "left_rows": left_total,
            "right_rows": right_total,
            "matched_rows": matched_rows_local,
            "pct_of_left": (100.0 * matched_rows_local / left_total) if left_total else 100.0,
            "pct_of_right": (100.0 * matched_rows_local / right_total) if right_total else 100.0,
            "only_left_rows": only_left,
            "only_right_rows": only_right,
            "exact_full_hash_match": left_h["row_hash"].sort().equals(right_h["row_hash"].sort()),
        }

    def duplicate_rows(df: pl.DataFrame) -> int:
        h = add_row_hash(df)
        dup = h.group_by("row_hash").len().filter(pl.col("len") > 1)
        return int((dup["len"].sum() - dup.height) if dup.height else 0)

    # Align to same logical column names
    syn_aligned = synapse_full_table_df.select([
        pl.col(f"SYNAPSE_{c}").alias(c) for c in common_cols
    ])
    dbx_aligned = databricks_full_table_df.select([
        pl.col(f"DATABRICKS_{c}").alias(c) for c in common_cols
    ])
    dbx_dedup_aligned = databricks_dedup_full_table_df.select([
        pl.col(f"DATABRICKS_{c}").alias(c) for c in common_cols
    ])

    syn_norm = normalize_types(syn_aligned)
    dbx_norm = normalize_types(dbx_aligned)
    dbx_dedup_norm = normalize_types(dbx_dedup_aligned)

    # Baseline result used as core output
    baseline_stats = calc_hash_match_stats(syn_norm, dbx_norm)
    full_match_exact = baseline_stats["exact_full_hash_match"]

    syn_total = baseline_stats["left_rows"]
    dbx_total = baseline_stats["right_rows"]
    matched_rows = min(baseline_stats["matched_rows"], syn_total, dbx_total)

    pct_of_synapse = 100.0 * matched_rows / syn_total if syn_total else 100.0
    pct_of_databricks = 100.0 * matched_rows / dbx_total if dbx_total else 100.0
    pct_overall = 100.0 * matched_rows / max(syn_total, dbx_total) if max(syn_total, dbx_total) else 100.0

    diagnostics = {
        "hash_scenarios_df": pl.DataFrame([]),
        "duplicate_profile_df": pl.DataFrame([]),
        "key_overlap_df": pl.DataFrame([]),
        "key_multiplicity_df": pl.DataFrame([]),
        "float_columns": [],
        "string_columns": [],
    }

    if include_diagnostics:
        float_cols = [c for c, t in syn_norm.schema.items() if t in FLOAT_LIKE]
        string_cols = [c for c, t in syn_norm.schema.items() if t == pl.Utf8]

        scenario_rows = []

        def add_scenario(name: str, left_df: pl.DataFrame, right_df: pl.DataFrame):
            s = calc_hash_match_stats(left_df, right_df)
            scenario_rows.append({
                "scenario": name,
                "left_rows": s["left_rows"],
                "right_rows": s["right_rows"],
                "matched_rows": s["matched_rows"],
                "pct_of_left": s["pct_of_left"],
                "pct_of_right": s["pct_of_right"],
                "only_left_rows": s["only_left_rows"],
                "only_right_rows": s["only_right_rows"],
                "exact_full_hash_match": s["exact_full_hash_match"],
            })

        add_scenario("baseline", syn_norm, dbx_norm)
        add_scenario("trim_strings", normalize_strings(syn_norm, string_cols, "trim"), normalize_strings(dbx_norm, string_cols, "trim"))
        add_scenario("trim_upper_strings", normalize_strings(syn_norm, string_cols, "trim_upper"), normalize_strings(dbx_norm, string_cols, "trim_upper"))

        non_float_non_datetime_cols = [
            c for c, t in syn_norm.schema.items()
            if (t not in FLOAT_LIKE) and (not str(t).startswith("Datetime"))
        ]
        if non_float_non_datetime_cols:
            add_scenario(
                "baseline_excluding_datetime_float",
                syn_norm.select(non_float_non_datetime_cols),
                dbx_norm.select(non_float_non_datetime_cols),
            )

        add_scenario("databricks_dedup_by_rtlh_row_hash", syn_norm, dbx_dedup_norm)

        for d in float_round_decimals:
            add_scenario(
                f"round_float_{d}dp",
                round_float_cols(syn_norm, float_cols, d),
                round_float_cols(dbx_norm, float_cols, d),
            )

        hash_scenarios_df = pl.DataFrame(scenario_rows)

        duplicate_profile_df = pl.DataFrame([
            {
                "dataset": "synapse",
                "duplicate_rows": duplicate_rows(syn_norm),
                "total_rows": syn_norm.height,
            },
            {
                "dataset": "databricks",
                "duplicate_rows": duplicate_rows(dbx_norm),
                "total_rows": dbx_norm.height,
            },
        ]).with_columns(
            (100.0 * pl.col("duplicate_rows") / pl.col("total_rows")).alias("duplicate_pct")
        )

        if candidate_keys is None:
            candidate_keys = [
                ["TAGNAME", "DATEFROM", "DATETO"],
                ["TAGNAME", "YEARMONTH"],
                ["TAGNAME"],
            ]

        key_overlap_rows = []
        key_multiplicity_rows = []

        for key_cols in candidate_keys:
            if set(key_cols).issubset(set(common_cols)):
                syn_keys = syn_norm.select(key_cols).unique()
                dbx_keys = dbx_norm.select(key_cols).unique()

                syn_only = syn_keys.join(dbx_keys, on=key_cols, how="anti").height
                dbx_only = dbx_keys.join(syn_keys, on=key_cols, how="anti").height

                key_overlap_rows.append({
                    "key_name": " + ".join(key_cols),
                    "syn_distinct_keys": syn_keys.height,
                    "dbx_distinct_keys": dbx_keys.height,
                    "keys_only_in_synapse": syn_only,
                    "keys_only_in_databricks": dbx_only,
                })

                syn_key_cnt = syn_norm.group_by(key_cols).len().rename({"len": "syn_key_rows"})
                dbx_key_cnt = dbx_norm.group_by(key_cols).len().rename({"len": "dbx_key_rows"})
                key_cmp = syn_key_cnt.join(dbx_key_cnt, on=key_cols, how="inner")

                key_multiplicity_rows.append({
                    "key_name": " + ".join(key_cols),
                    "keys_where_synapse_has_duplicates": key_cmp.filter(pl.col("syn_key_rows") > 1).height,
                    "keys_where_databricks_has_duplicates": key_cmp.filter(pl.col("dbx_key_rows") > 1).height,
                    "keys_where_databricks_has_more_rows": key_cmp.filter(pl.col("dbx_key_rows") > pl.col("syn_key_rows")).height,
                    "extra_databricks_rows_by_key_multiplicity": int(
                        key_cmp
                        .with_columns((pl.col("dbx_key_rows") - pl.col("syn_key_rows")).clip(lower_bound=0).alias("extra"))["extra"]
                        .sum() or 0
                    ),
                })

        key_overlap_df = pl.DataFrame(key_overlap_rows) if key_overlap_rows else pl.DataFrame([])
        key_multiplicity_df = pl.DataFrame(key_multiplicity_rows) if key_multiplicity_rows else pl.DataFrame([])

        diagnostics = {
            "hash_scenarios_df": hash_scenarios_df,
            "duplicate_profile_df": duplicate_profile_df,
            "key_overlap_df": key_overlap_df,
            "key_multiplicity_df": key_multiplicity_df,
            "float_columns": float_cols,
            "string_columns": string_cols,
        }

    return {
        "schema_matches": schema_matches,
        "row_count_matches": row_count_matches,
        "full_match": full_match_exact,
        "synapse_row_count": synapse_row_count,
        "databricks_row_count": databricks_row_count,
        "schema_mismatches_df": schema_mismatches_df,
        "schema_comparison_df": schema_comparison_df,
        "common_columns_compared": common_cols,
        "matched_rows": matched_rows,
        "syn_total": syn_total,
        "dbx_total": dbx_total,
        "pct_of_synapse": pct_of_synapse,
        "pct_of_databricks": pct_of_databricks,
        "pct_overall": pct_overall,
        "diagnostics": diagnostics,
    }

In [4]:
# opsdata_mis_rtit_headgrade
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="hstg_v",
    synapse_table_name="vw_hstg_udlpisystems_opsdata_mis_rtit_headgrade",
    databricks_catalog_name="lakehouse",
    databricks_schema_name="analytics_and_enablement_staging_pi_systems",
    databricks_table_name="opsdata_mis_rtit_headgrade",
    candidate_keys=[["DATETO", "RESOLUTIONCODE", "TAGNAME"]],
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Row Count Matches: {result['row_count_matches']}")
print(f"Exact Full Hash Match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")

print("\nSchema comparison")
display(result["schema_comparison_df"])

print("\nHash scenarios")
display(result["diagnostics"]["hash_scenarios_df"])

print("\nDuplicate profile")
display(result["diagnostics"]["duplicate_profile_df"])

print("\nKey overlap")
display(result["diagnostics"]["key_overlap_df"])

print("\nKey multiplicity")
display(result["diagnostics"]["key_multiplicity_df"])


Schema Matches: True
Row Count Matches: True
Exact Full Hash Match: True
Synapse row count: 12442
Databricks row count: 12442
Matched: 12442/12442 (100.00000%)

Schema comparison


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""ACTUAL""","""FLOAT""","""ACTUAL""","""DOUBLE""","""FLOAT"""
"""BUDGET""","""FLOAT""","""BUDGET""","""DOUBLE""","""FLOAT"""
"""DATEFROM""","""DATETIME2""","""DATEFROM""","""TIMESTAMP""","""DATETIME2"""
"""DATETO""","""DATETIME2""","""DATETO""","""TIMESTAMP""","""DATETIME2"""
"""FORECAST""","""FLOAT""","""FORECAST""","""DOUBLE""","""FLOAT"""
"""RESOLUTIONCODE""","""VARCHAR""","""RESOLUTIONCODE""","""STRING""","""VARCHAR"""
"""RESOLUTIONDESC""","""VARCHAR""","""RESOLUTIONDESC""","""STRING""","""VARCHAR"""
"""SERVERTIMEZONE""","""VARCHAR""","""SERVERTIMEZONE""","""STRING""","""VARCHAR"""
"""SITECODE""","""VARCHAR""","""SITECODE""","""STRING""","""VARCHAR"""



Hash scenarios


scenario,left_rows,right_rows,matched_rows,pct_of_left,pct_of_right,only_left_rows,only_right_rows,exact_full_hash_match
str,i64,i64,i64,f64,f64,i64,i64,bool
"""baseline""",12442,12442,12442,100.0,100.0,0,0,true
"""trim_strings""",12442,12442,12442,100.0,100.0,0,0,true
"""trim_upper_strings""",12442,12442,12442,100.0,100.0,0,0,true
"""baseline_excluding_datetime_float""",12442,12442,12442,100.0,100.0,0,0,true
"""databricks_dedup_by_rtlh_row_hash""",12442,12442,12442,100.0,100.0,0,0,true
"""round_float_6dp""",12442,12442,12442,100.0,100.0,0,0,true
"""round_float_3dp""",12442,12442,12442,100.0,100.0,0,0,true



Duplicate profile


dataset,duplicate_rows,total_rows,duplicate_pct
str,i64,i64,f64
"""synapse""",0,12442,0.0
"""databricks""",0,12442,0.0



Key overlap


key_name,syn_distinct_keys,dbx_distinct_keys,keys_only_in_synapse,keys_only_in_databricks
str,i64,i64,i64,i64
"""DATETO + RESOLUTIONCODE + TAGNAME""",12442,12442,0,0



Key multiplicity


key_name,keys_where_synapse_has_duplicates,keys_where_databricks_has_duplicates,keys_where_databricks_has_more_rows,extra_databricks_rows_by_key_multiplicity
str,i64,i64,i64,i64
"""DATETO + RESOLUTIONCODE + TAGNAME""",0,0,0,0


In [5]:
# opsdata_mis_rtit_sandmetrics
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="hstg_v",
    synapse_table_name="vw_hstg_udlpisystems_opsdata_cs_mis_rtit_sandmetrics",
    databricks_catalog_name="lakehouse",
    databricks_schema_name="analytics_and_enablement_staging_pi_systems",
    databricks_table_name="opsdata_mis_rtit_sandmetrics",
    candidate_keys=[["TAGNAME", "DATEFROM", "DATETO"]],
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Row Count Matches: {result['row_count_matches']}")
print(f"Exact Full Hash Match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")

print("\nSchema comparison")
display(result["schema_comparison_df"])

print("\nHash scenarios")
display(result["diagnostics"]["hash_scenarios_df"])

print("\nDuplicate profile")
display(result["diagnostics"]["duplicate_profile_df"])

print("\nKey overlap")
display(result["diagnostics"]["key_overlap_df"])

print("\nKey multiplicity")
display(result["diagnostics"]["key_multiplicity_df"])


Schema Matches: True
Row Count Matches: False
Exact Full Hash Match: False
Synapse row count: 86299
Databricks row count: 99154
Matched: 19706/86299 (22.83456%)

Schema comparison


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""ACTUAL""","""FLOAT""","""ACTUAL""","""DOUBLE""","""FLOAT"""
"""BUDGET""","""FLOAT""","""BUDGET""","""DOUBLE""","""FLOAT"""
"""DATEFROM""","""DATETIME2""","""DATEFROM""","""TIMESTAMP""","""DATETIME2"""
"""DATETO""","""DATETIME2""","""DATETO""","""TIMESTAMP""","""DATETIME2"""
"""FORECAST""","""FLOAT""","""FORECAST""","""DOUBLE""","""FLOAT"""
"""RESOLUTIONCODE""","""VARCHAR""","""RESOLUTIONCODE""","""STRING""","""VARCHAR"""
"""RESOLUTIONDESC""","""VARCHAR""","""RESOLUTIONDESC""","""STRING""","""VARCHAR"""
"""SERVERTIMEZONE""","""VARCHAR""","""SERVERTIMEZONE""","""STRING""","""VARCHAR"""
"""SITECODE""","""VARCHAR""","""SITECODE""","""STRING""","""VARCHAR"""



Hash scenarios


scenario,left_rows,right_rows,matched_rows,pct_of_left,pct_of_right,only_left_rows,only_right_rows,exact_full_hash_match
str,i64,i64,i64,f64,f64,i64,i64,bool
"""baseline""",86299,99154,19706,22.834564,19.874135,66593,79448,false
"""trim_strings""",86299,99154,19706,22.834564,19.874135,66593,79448,false
"""trim_upper_strings""",86299,99154,19706,22.834564,19.874135,66593,79448,false
"""baseline_excluding_datetime_float""",86299,99154,86299,100.0,87.035319,0,12855,false
"""databricks_dedup_by_rtlh_row_hash""",86299,85569,18985,21.999096,22.186773,67314,66584,false
"""round_float_6dp""",86299,99154,20534,23.794018,20.7092,65765,78620,false
"""round_float_3dp""",86299,99154,39244,45.474455,39.578837,47055,59910,false



Duplicate profile


dataset,duplicate_rows,total_rows,duplicate_pct
str,i64,i64,f64
"""synapse""",0,86299,0.0
"""databricks""",12855,99154,12.964681



Key overlap


key_name,syn_distinct_keys,dbx_distinct_keys,keys_only_in_synapse,keys_only_in_databricks
str,i64,i64,i64,i64
"""TAGNAME + DATEFROM + DATETO""",31387,31387,0,0



Key multiplicity


key_name,keys_where_synapse_has_duplicates,keys_where_databricks_has_duplicates,keys_where_databricks_has_more_rows,extra_databricks_rows_by_key_multiplicity
str,i64,i64,i64,i64
"""TAGNAME + DATEFROM + DATETO""",21476,24415,6421,12855


In [6]:
# opsdata_pm_reporting_data
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="hstg_v",
    synapse_table_name="vw_hstg_udlpisystems_opsdata_cs_pm_reporting_data",
    databricks_catalog_name="lakehouse",
    databricks_schema_name="analytics_and_enablement_staging_pi_systems",
    databricks_table_name="opsdata_pm_reporting_data",
    candidate_keys=[["ELEMENTATTRIBUTE", "ENDTIMELOCAL", "YEARMONTH"]],
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Row Count Matches: {result['row_count_matches']}")
print(f"Exact Full Hash Match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")

print("\nSchema comparison")
display(result["schema_comparison_df"])

print("\nHash scenarios")
display(result["diagnostics"]["hash_scenarios_df"])

print("\nDuplicate profile")
display(result["diagnostics"]["duplicate_profile_df"])

print("\nKey overlap")
display(result["diagnostics"]["key_overlap_df"])

print("\nKey multiplicity")
display(result["diagnostics"]["key_multiplicity_df"])


Schema Matches: True
Row Count Matches: False
Exact Full Hash Match: False
Synapse row count: 149547
Databricks row count: 157938
Matched: 25752/149547 (17.22000%)

Schema comparison


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""CT""","""FLOAT""","""CT""","""DOUBLE""","""FLOAT"""
"""CU""","""FLOAT""","""CU""","""DOUBLE""","""FLOAT"""
"""DYNAMIC_MDR""","""FLOAT""","""DYNAMIC_MDR""","""DOUBLE""","""FLOAT"""
"""EHNAME""","""VARCHAR""","""EHNAME""","""STRING""","""VARCHAR"""
"""EHPATH""","""VARCHAR""","""EHPATH""","""STRING""","""VARCHAR"""
"""ELEMENTATTRIBUTE""","""VARCHAR""","""ELEMENTATTRIBUTE""","""STRING""","""VARCHAR"""
"""ENDTIMELOCAL""","""DATETIME2""","""ENDTIMELOCAL""","""TIMESTAMP""","""DATETIME2"""
"""ENDTIMEUTC""","""DATETIME2""","""ENDTIMEUTC""","""TIMESTAMP""","""DATETIME2"""
"""EU_METRIC""","""FLOAT""","""EU_METRIC""","""DOUBLE""","""FLOAT"""



Hash scenarios


scenario,left_rows,right_rows,matched_rows,pct_of_left,pct_of_right,only_left_rows,only_right_rows,exact_full_hash_match
str,i64,i64,i64,f64,f64,i64,i64,bool
"""baseline""",149547,157938,25752,17.220004,16.305132,123795,132186,false
"""trim_strings""",149547,157938,25752,17.220004,16.305132,123795,132186,false
"""trim_upper_strings""",149547,157938,25752,17.220004,16.305132,123795,132186,false
"""baseline_excluding_datetime_float""",149547,157938,142315,95.164062,90.108144,7232,15623,false
"""databricks_dedup_by_rtlh_row_hash""",149547,155746,25752,17.220004,16.534614,123795,129994,false
"""round_float_6dp""",149547,157938,31983,21.386587,20.250351,117564,125955,false
"""round_float_3dp""",149547,157938,83461,55.80921,52.844154,66086,74477,false



Duplicate profile


dataset,duplicate_rows,total_rows,duplicate_pct
str,i64,i64,f64
"""synapse""",0,149547,0.0
"""databricks""",2192,157938,1.387886



Key overlap


key_name,syn_distinct_keys,dbx_distinct_keys,keys_only_in_synapse,keys_only_in_databricks
str,i64,i64,i64,i64
"""ELEMENTATTRIBUTE + ENDTIMELOCAL + YEARMONTH""",149547,155746,7232,13431



Key multiplicity


key_name,keys_where_synapse_has_duplicates,keys_where_databricks_has_duplicates,keys_where_databricks_has_more_rows,extra_databricks_rows_by_key_multiplicity
str,i64,i64,i64,i64
"""ELEMENTATTRIBUTE + ENDTIMELOCAL + YEARMONTH""",0,186,186,186


In [7]:
# opsdata_pm_reporting_recovery_data
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="hstg_v",
    synapse_table_name="vw_hstg_udlpisystems_opsdata_cs_pm_reporting_recovery_data",
    databricks_catalog_name="lakehouse",
    databricks_schema_name="analytics_and_enablement_staging_pi_systems",
    databricks_table_name="opsdata_pm_reporting_recovery_data",
    candidate_keys=[["ELEMENTATTRIBUTE", "STARTTIMEUTC", "YEARMONTH"]],
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Row Count Matches: {result['row_count_matches']}")
print(f"Exact Full Hash Match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")

print("\nSchema comparison")
display(result["schema_comparison_df"])

print("\nHash scenarios")
display(result["diagnostics"]["hash_scenarios_df"])

print("\nDuplicate profile")
display(result["diagnostics"]["duplicate_profile_df"])

print("\nKey overlap")
display(result["diagnostics"]["key_overlap_df"])

print("\nKey multiplicity")
display(result["diagnostics"]["key_multiplicity_df"])


Schema Matches: True
Row Count Matches: False
Exact Full Hash Match: False
Synapse row count: 2471
Databricks row count: 2626
Matched: 2430/2471 (98.34075%)

Schema comparison


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""CT""","""FLOAT""","""CT""","""DOUBLE""","""FLOAT"""
"""EHNAME""","""VARCHAR""","""EHNAME""","""STRING""","""VARCHAR"""
"""EHPATH""","""VARCHAR""","""EHPATH""","""STRING""","""VARCHAR"""
"""ELEMENTATTRIBUTE""","""VARCHAR""","""ELEMENTATTRIBUTE""","""STRING""","""VARCHAR"""
"""ENDTIMELOCAL""","""DATETIME2""","""ENDTIMELOCAL""","""TIMESTAMP""","""DATETIME2"""
"""ENDTIMEUTC""","""DATETIME2""","""ENDTIMEUTC""","""TIMESTAMP""","""DATETIME2"""
"""PRODUCTIONOFFSET""","""FLOAT""","""PRODUCTIONOFFSET""","""DOUBLE""","""FLOAT"""
"""RECOVERY""","""FLOAT""","""RECOVERY""","""DOUBLE""","""FLOAT"""
"""RECOVERY_BUDGET""","""FLOAT""","""RECOVERY_BUDGET""","""DOUBLE""","""FLOAT"""



Hash scenarios


scenario,left_rows,right_rows,matched_rows,pct_of_left,pct_of_right,only_left_rows,only_right_rows,exact_full_hash_match
str,i64,i64,i64,f64,f64,i64,i64,bool
"""baseline""",2471,2626,2430,98.340753,92.536177,41,196,false
"""trim_strings""",2471,2626,2430,98.340753,92.536177,41,196,false
"""trim_upper_strings""",2471,2626,2430,98.340753,92.536177,41,196,false
"""baseline_excluding_datetime_float""",2471,2626,2460,99.554836,93.678599,11,166,false
"""databricks_dedup_by_rtlh_row_hash""",2471,2527,2430,98.340753,96.161456,41,97,false
"""round_float_6dp""",2471,2626,2444,98.907325,93.069307,27,182,false
"""round_float_3dp""",2471,2626,2444,98.907325,93.069307,27,182,false



Duplicate profile


dataset,duplicate_rows,total_rows,duplicate_pct
str,i64,i64,f64
"""synapse""",0,2471,0.0
"""databricks""",99,2626,3.769992



Key overlap


key_name,syn_distinct_keys,dbx_distinct_keys,keys_only_in_synapse,keys_only_in_databricks
str,i64,i64,i64,i64
"""ELEMENTATTRIBUTE + STARTTIMEUTC + YEARMONTH""",2456,2512,11,67



Key multiplicity


key_name,keys_where_synapse_has_duplicates,keys_where_databricks_has_duplicates,keys_where_databricks_has_more_rows,extra_databricks_rows_by_key_multiplicity
str,i64,i64,i64,i64
"""ELEMENTATTRIBUTE + STARTTIMEUTC + YEARMONTH""",15,71,59,99


In [8]:
# globalpi_pm_reporting_locations_hierarchy
result = compare_synapse_vs_databricks(
    databricks_conn=databricks_conn,
    synapse_conn=synapse_conn,
    synapse_schema_name="hstg_v",
    synapse_table_name="vw_hstg_udlpisystems_pm_reporting_locations_hierarchy",
    databricks_catalog_name="lakehouse",
    databricks_schema_name="analytics_and_enablement_staging_pi_systems",
    databricks_table_name="globalpi_pm_reporting_locations_hierarchy",
    candidate_keys=[["EANAME", "EAPATH", "ELEMENTATTRIBUTE"]],
)

print(f"Schema Matches: {result['schema_matches']}")
print(f"Row Count Matches: {result['row_count_matches']}")
print(f"Exact Full Hash Match: {result['full_match']}")
print(f"Synapse row count: {result['synapse_row_count']}")
print(f"Databricks row count: {result['databricks_row_count']}")
print(f"Matched: {result['matched_rows']}/{result['syn_total']} ({result['pct_of_synapse']:.5f}%)")

print("\nSchema comparison")
display(result["schema_comparison_df"])

print("\nHash scenarios")
display(result["diagnostics"]["hash_scenarios_df"])

print("\nDuplicate profile")
display(result["diagnostics"]["duplicate_profile_df"])

print("\nKey overlap")
display(result["diagnostics"]["key_overlap_df"])

print("\nKey multiplicity")
display(result["diagnostics"]["key_multiplicity_df"])


Schema Matches: False
Row Count Matches: True
Exact Full Hash Match: True
Synapse row count: 3318
Databricks row count: 3318
Matched: 3318/3318 (100.00000%)

Schema comparison


column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""EANAME""","""VARCHAR""","""EANAME""","""STRING""","""VARCHAR"""
"""EAPATH""","""VARCHAR""","""EAPATH""","""STRING""","""VARCHAR"""
"""EHNAME""","""VARCHAR""","""EHNAME""","""STRING""","""VARCHAR"""
"""EHPATH""","""VARCHAR""","""EHPATH""","""STRING""","""VARCHAR"""
"""ELEMENTATTRIBUTE""","""VARCHAR""","""ELEMENTATTRIBUTE""","""STRING""","""VARCHAR"""
"""SOURCE_TAG""","""VARCHAR""","""SOURCE_TAG""","""STRING""","""VARCHAR"""
null,null,"""TIME""","""TIMESTAMP""","""DATETIME2"""
"""UOM""","""VARCHAR""","""UOM""","""STRING""","""VARCHAR"""
null,null,"""VALUE""","""STRING""","""VARCHAR"""



Hash scenarios


scenario,left_rows,right_rows,matched_rows,pct_of_left,pct_of_right,only_left_rows,only_right_rows,exact_full_hash_match
str,i64,i64,i64,f64,f64,i64,i64,bool
"""baseline""",3318,3318,3318,100.0,100.0,0,0,true
"""trim_strings""",3318,3318,3318,100.0,100.0,0,0,true
"""trim_upper_strings""",3318,3318,3318,100.0,100.0,0,0,true
"""baseline_excluding_datetime_float""",3318,3318,3318,100.0,100.0,0,0,true
"""databricks_dedup_by_rtlh_row_hash""",3318,3318,3318,100.0,100.0,0,0,true
"""round_float_6dp""",3318,3318,3318,100.0,100.0,0,0,true
"""round_float_3dp""",3318,3318,3318,100.0,100.0,0,0,true



Duplicate profile


dataset,duplicate_rows,total_rows,duplicate_pct
str,i64,i64,f64
"""synapse""",0,3318,0.0
"""databricks""",0,3318,0.0



Key overlap


key_name,syn_distinct_keys,dbx_distinct_keys,keys_only_in_synapse,keys_only_in_databricks
str,i64,i64,i64,i64
"""EANAME + EAPATH + ELEMENTATTRIBUTE""",3318,3318,0,0



Key multiplicity


key_name,keys_where_synapse_has_duplicates,keys_where_databricks_has_duplicates,keys_where_databricks_has_more_rows,extra_databricks_rows_by_key_multiplicity
str,i64,i64,i64,i64
"""EANAME + EAPATH + ELEMENTATTRIBUTE""",0,0,0,0


# Synapse vs Databricks Comparison Summary

Source notebook: check_source_tables_pi.ipynb  
Generated on: 2026-06-08

## Scope
This summary is based only on the executed comparison cells in check_source_tables_pi.ipynb.

## Executive Summary
- 5 table comparisons were run.
- Schema comparison passed for 4/5 tables.
- Row count comparison passed for 2/5 tables.
- Exact full-row hash match passed for 2/5 tables.
- Databricks duplicates are concentrated in 3 tables and explain most row count deltas.
- The new `baseline_excluding_datetime_float` scenario shows much higher alignment, indicating datetime/float precision is a major source of mismatch on some tables.

## Table-by-Table Core Results

| # | Synapse Table | Databricks Table | Schema Matches | Row Count Matches | Full Hash Match | Synapse Rows | Databricks Rows | Matched Rows | Match % of Synapse |
|---|---|---|---|---|---|---:|---:|---:|---:|
| 1 | vw_hstg_udlpisystems_opsdata_mis_rtit_headgrade | opsdata_mis_rtit_headgrade | True | True | True | 12442 | 12442 | 12442 | 100.00000% |
| 2 | vw_hstg_udlpisystems_opsdata_cs_mis_rtit_sandmetrics | opsdata_mis_rtit_sandmetrics | True | False | False | 86299 | 99154 | 19706 | 22.83456% |
| 3 | vw_hstg_udlpisystems_opsdata_cs_pm_reporting_data | opsdata_pm_reporting_data | True | False | False | 149547 | 157938 | 25752 | 17.22000% |
| 4 | vw_hstg_udlpisystems_opsdata_cs_pm_reporting_recovery_data | opsdata_pm_reporting_recovery_data | True | False | False | 2471 | 2626 | 2430 | 98.34075% |
| 5 | vw_hstg_udlpisystems_pm_reporting_locations_hierarchy | globalpi_pm_reporting_locations_hierarchy | False | True | True | 3318 | 3318 | 3318 | 100.00000% |

## New Scenario: Baseline Excluding Datetime/Float

| Table | Matched Rows | Synapse Rows | Match % of Synapse |
|---|---:|---:|---:|
| opsdata_mis_rtit_headgrade | 12442 | 12442 | 100.00000% |
| opsdata_mis_rtit_sandmetrics | 86299 | 86299 | 100.00000% |
| opsdata_pm_reporting_data | 142315 | 149547 | 95.16406% |
| opsdata_pm_reporting_recovery_data | 2460 | 2471 | 99.55484% |
| globalpi_pm_reporting_locations_hierarchy | 3318 | 3318 | 100.00000% |

## New Scenario: Databricks Dedup by `rtlh_row_hash`

Dedup logic used:
```sql
ROW_NUMBER() OVER (
  PARTITION BY rtlh_row_hash
  ORDER BY rtlh_ingestion_time DESC
) AS rn
```

| Table | Dedup Databricks Rows | Matched Rows | Match % of Synapse |
|---|---:|---:|---:|
| opsdata_mis_rtit_headgrade | 12442 | 12442 | 100.00000% |
| opsdata_mis_rtit_sandmetrics | 85569 | 18985 | 21.99910% |
| opsdata_pm_reporting_data | 155746 | 25752 | 17.22000% |
| opsdata_pm_reporting_recovery_data | 2527 | 2430 | 98.34075% |
| globalpi_pm_reporting_locations_hierarchy | 3318 | 3318 | 100.00000% |

## Duplicate Profile (from diagnostics)

| Table | Synapse Duplicate Rows | Databricks Duplicate Rows | Databricks Duplicate % |
|---|---:|---:|---:|
| opsdata_mis_rtit_headgrade | 0 | 0 | 0.000000% |
| opsdata_mis_rtit_sandmetrics | 0 | 12855 | 12.964681% |
| opsdata_pm_reporting_data | 0 | 2192 | 1.387886% |
| opsdata_pm_reporting_recovery_data | 0 | 99 | 3.769992% |
| globalpi_pm_reporting_locations_hierarchy | 0 | 0 | 0.000000% |

## Key Findings
- `opsdata_mis_rtit_headgrade` is fully aligned across schema, row count, and full hash.
- `opsdata_mis_rtit_sandmetrics` has a large Databricks overcount (+12855) driven by duplicates, while `baseline_excluding_datetime_float` reaches 100% match of Synapse.
- `opsdata_pm_reporting_data` has Databricks overcount (+8391), key-set differences, and float/datetime sensitivity (`baseline_excluding_datetime_float` improves to 95.16%).
- `opsdata_pm_reporting_recovery_data` is close to parity on content (98.34%), with a small Databricks overcount (+155) and 99 duplicate rows.
- `globalpi_pm_reporting_locations_hierarchy` has full row/hash parity, but schema mismatch persists due to `TIME`/`VALUE` vs `TIME_RIO`/`VALUE_RIO`.
- Schema mismatch details:
  - Present in Databricks only: TIME, VALUE
  - Present in Synapse only: TIME_RIO, VALUE_RIO

## Candidate Key Diagnostics Snapshot
- opsdata_mis_rtit_headgrade (DATETO + RESOLUTIONCODE + TAGNAME): perfect overlap and no multiplicity issues.
- opsdata_mis_rtit_sandmetrics (TAGNAME + DATEFROM + DATETO): distinct key sets match, but Databricks has higher multiplicity (extra rows by key multiplicity: 12855).
- opsdata_pm_reporting_data (ELEMENTATTRIBUTE + ENDTIMELOCAL + YEARMONTH): key-set differences exist in both directions and Databricks has extra multiplicity (186).
- opsdata_pm_reporting_recovery_data (ELEMENTATTRIBUTE + STARTTIMEUTC + YEARMONTH): small key-set differences both directions and extra multiplicity in Databricks (99).
- globalpi_pm_reporting_locations_hierarchy (EANAME + EAPATH + ELEMENTATTRIBUTE): perfect overlap and no multiplicity issues.

## Recommended Actions
1. Deduplicate Databricks rows by `rtlh_row_hash` for sandmetrics, pm_reporting_data, and pm_reporting_recovery_data.
2. For sandmetrics and pm_reporting_data, confirm precision policy for datetime/float columns between Synapse and Databricks.
3. Align schema naming for hierarchy table (`TIME`/`VALUE` vs `TIME_RIO`/`VALUE_RIO`) to remove false schema mismatch while preserving data parity.
4. Re-run comparison after any transformation changes to confirm row count and full-hash convergence.
